# FloodLink — เทรน WangchanBERTa ด้วยข้อมูลจริง (Colab)

**ก่อนเริ่ม:** เมนู **Runtime → Change runtime type → T4 GPU** แล้วกด **Runtime → Run all**

ต้องมีไฟล์ 3 ไฟล์จาก `04_flood_nlp_model/realdata/data/` (สร้างด้วย `build_dataset.py`):
`train.jsonl`, `dev.jsonl`, `test.jsonl` — ถ้ายังไม่ได้วางใน Google Drive ช่องที่ 2 จะให้อัปโหลด

| ขั้น | เวลาโดยประมาณ (T4) |
|---|---|
| ติดตั้ง + โหลดโมเดล | 2–3 นาที |
| เทรน (สูงสุด 8 รอบ หยุดเองเมื่อไม่ดีขึ้น) | 10–25 นาที |
| วัดผล + ส่งออก | 2 นาที |

ผลลัพธ์: `floodwcb_model.zip` → แตกไฟล์ไว้ที่ `04_flood_nlp_model/artifacts/floodwcb_model/` ในโปรเจกต์

## 1. เช็ก GPU และติดตั้ง

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "⚠️ ไม่มี GPU — ไปที่ Runtime > Change runtime type > T4 GPU"
!pip -q install sentencepiece
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())

## 2. ตั้งค่า + ข้อมูล
ถ้าใช้ Google Drive: วางไฟล์ไว้ที่ `MyDrive/floodlink_wcb/data/` แล้วโมเดลจะถูกเซฟลง Drive ด้วย (กันหายถ้า Colab ตัด)

In [ ]:
from pathlib import Path
import shutil, json, os

USE_DRIVE = True          # False = ไม่ใช้ Drive (อัปโหลดเอง, เซฟใน Colab อย่างเดียว)
MAX_LEN = 256             # ข้อความยาวกว่านี้ (ราว 5% ของข้อมูล) จะถูกตัดท้าย
EPOCHS, BATCH, LR = 8, 16, 3e-5
PATIENCE = 2              # หยุดเมื่อคะแนน dev ไม่ดีขึ้น 2 รอบติด
RED_RECALL_TARGET = 0.98

WORK = Path("/content/work"); WORK.mkdir(exist_ok=True)
DATA = Path("/content/data"); DATA.mkdir(exist_ok=True)
DRIVE_DIR = None
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_DIR = Path("/content/drive/MyDrive/floodlink_wcb"); (DRIVE_DIR / "data").mkdir(parents=True, exist_ok=True)
        for n in ("train.jsonl", "dev.jsonl", "test.jsonl"):
            if (DRIVE_DIR / "data" / n).exists():
                shutil.copy(DRIVE_DIR / "data" / n, DATA / n)
    except Exception as e:
        print("ใช้ Drive ไม่ได้:", e); DRIVE_DIR = None

need = ["train.jsonl", "dev.jsonl", "test.jsonl"]
if not all((DATA / n).exists() for n in need):
    from google.colab import files
    print("อัปโหลด train.jsonl, dev.jsonl, test.jsonl (เลือกพร้อมกันได้)")
    for name, b in files.upload().items():
        for n in need:                                   # "train (1).jsonl" -> train.jsonl
            if name.replace(" ", "").startswith(n.split(".")[0]):
                (DATA / n).write_bytes(b)
missing = [n for n in need if not (DATA / n).exists()]
assert not missing, f"ยังขาดไฟล์: {missing}"
print("ข้อมูลพร้อม:", [f"{n} {sum(1 for _ in open(DATA / n, encoding='utf-8'))} แถว" for n in need])

## 3. โค้ดโมเดล (`floodwcb.py` — ไฟล์เดียวกับในโปรเจกต์)

In [ ]:
%%writefile floodwcb.py
"""FloodLink WangchanBERTa model (multi-task) — shared by the Colab notebook and the backend.

One encoder, six heads, trained together on the real Hat Yai data (realdata/):
  type   : help_request / status_update_resolved / hazard_report / donation_offer / irrelevant   (softmax)
  flags  : vulnerable groups, multi-label                                                       (sigmoid)
  needs  : what they need, multi-label                                                          (sigmoid)
  water  : water level (or none)                                                                (softmax)
  bin    : lost_contact, insufficient_info                                                      (sigmoid)
  prio   : direct priority guess (help requests only) — a safety net next to the rule colour    (softmax)

The final colour = the MORE urgent of (rules applied to the predicted facts, priority head), so a miss in one
path can be caught by the other. Thresholds for life-threatening flags are tuned for recall, not F1.

Pure-python/numpy parts (preprocess, labels, decode, thresholds, metrics) work without torch.
"""
import json, math, re
from pathlib import Path

import numpy as np

MODEL_NAME = "airesearch/wangchanberta-base-att-spm-uncased"
TYPES = ["help_request", "status_update_resolved", "hazard_report", "donation_offer", "irrelevant"]
FLAGS = ["bedridden", "wheelchair", "disabled_blind", "dialysis", "heart", "pregnant", "in_labor", "injured",
         "chronic_illness", "sick_now", "cannot_swim"]
NEEDS = ["evacuation", "food", "water", "milk", "medicine", "power", "medical"]
WATER = ["none", "knee", "waist", "chest", "floor1", "floor2", "roof"]
BIN = ["lost_contact", "insufficient_info"]
PRIO = ["red", "orange", "yellow", "green"]
RED_FLAGS = {"dialysis", "heart", "injured", "in_labor", "sick_now"}      # tuned for recall
SEV = {"knee": 1, "waist": 2, "chest": 3, "floor1": 3, "floor2": 4, "roof": 5}
RANK = {"red": 3, "orange": 2, "yellow": 1, "green": 0, "none": -1}
IGNORE = -100


# ---------------- text ----------------
def preprocess(text: str) -> str:
    """WangchanBERTa (thai2transformers) style: lowercase, collapse spaces, space -> <_>, cut long repeats."""
    t = text.replace("￼", " ").lower()
    t = re.sub(r"https?://\S+", " <url> ", t)
    t = re.sub(r"(.)\1{3,}", r"\1\1\1", t)                       # ค่าาาาาา -> ค่าาา, 🙏🙏🙏🙏🙏 -> 🙏🙏🙏
    t = re.sub(r"[ \t\r\n]+", " ", t).strip()
    return t.replace(" ", "<_>")


AGE_KID = re.compile(r"(\d{1,2}(?:\.\d)?)\s*(ขวบ|เดือน|ด\.)")
AGE_OLD = re.compile(r"(?:อายุ\s*|ยาย\s*|ตา\s*|ปู่\s*|ย่า\s*|แม่\s*|พ่อ\s*)(\d{2,3})(?:\s*ปี)?|(\d{2,3})\s*ปี")


def ages(text: str) -> tuple[list[float], list[int]]:
    """Light age extractor for the rules (the backend may use floodnlp.entities instead)."""
    kids = []
    for n, unit in AGE_KID.findall(text):
        v = float(n) / (12 if unit in ("เดือน", "ด.") else 1)
        if v < 18:
            kids.append(round(v, 2))
    old = []
    for a, b in AGE_OLD.findall(text):
        v = int(a or b)
        if 60 <= v <= 110:
            old.append(v)
    return kids, old


# ---------------- labels ----------------
def encode(row: dict) -> dict:
    """jsonl row from realdata/build_dataset.py -> numeric targets."""
    help_ = row["message_type"] == "help_request"
    wl = row.get("water_level") or "none"
    return {
        "type": TYPES.index(row["message_type"]),
        "flags": np.array([f in row.get("flags", []) for f in FLAGS], dtype=np.float32),
        "needs": np.array([n in row.get("needs", []) for n in NEEDS], dtype=np.float32),
        "water": WATER.index(wl) if (help_ and wl in WATER) else IGNORE,
        "bin": np.array([bool(row.get("lost_contact")), bool(row.get("insufficient_info"))], dtype=np.float32),
        "bin_mask": 1.0 if help_ else 0.0,
        "prio": PRIO.index(row["priority"]) if (help_ and row.get("priority") in PRIO) else IGNORE,
    }


def rule_priority(p: dict) -> str:
    """Same rules as realdata/build_dataset.rule_priority (and floodcommon.triage, extended)."""
    if p["message_type"] != "help_request":
        return "none"
    f, sev = set(p["flags"]), SEV.get(p.get("water_level"), -1)
    kids, old = p.get("children_ages") or [], p.get("elderly_ages") or []
    if f & RED_FLAGS or any(a < 1 for a in kids) or sev >= 5:
        return "red"
    if ("bedridden" in f and sev >= 4) or ("pregnant" in f and sev >= 3):
        return "red"
    if f & {"bedridden", "wheelchair", "disabled_blind", "pregnant"} or any(a >= 70 for a in old) or any(a < 6 for a in kids) or sev >= 4:
        return "orange"
    if sev >= 2 or {"food", "water", "medicine"} & set(p.get("needs") or []):
        return "yellow"
    return "green"


def _sigmoid(x):
    return 1 / (1 + np.exp(-x))


def _softmax(x):
    e = np.exp(x - x.max(-1, keepdims=True))
    return e / e.sum(-1, keepdims=True)


def to_probs(logits: dict) -> dict:
    """numpy logits (batch x k) -> probabilities."""
    return {"type": _softmax(logits["type"]), "flags": _sigmoid(logits["flags"]), "needs": _sigmoid(logits["needs"]),
            "water": _softmax(logits["water"]), "bin": _sigmoid(logits["bin"]), "prio": _softmax(logits["prio"])}


def default_thresholds() -> dict:
    return {"flags": {f: 0.5 for f in FLAGS}, "needs": {n: 0.5 for n in NEEDS}, "bin": {b: 0.5 for b in BIN}, "prio_red": 0.5,
            "help_min": 0.5}


def decode(probs: dict, i: int, th: dict, text: str = "") -> dict:
    """Probabilities of item i -> one prediction dict (facts + colour + confidence)."""
    t = TYPES[int(probs["type"][i].argmax())]
    flags = [f for j, f in enumerate(FLAGS) if probs["flags"][i, j] >= th["flags"][f]]
    overridden = False
    if t != "help_request" and set(flags) & RED_FLAGS and probs["type"][i, 0] >= th.get("help_min", 0.5):
        t, overridden = "help_request", True           # safety: a life-threatening signal must not be lost to a type mistake
    needs = [n for j, n in enumerate(NEEDS) if probs["needs"][i, j] >= th["needs"][n]]
    w = WATER[int(probs["water"][i].argmax())]
    kids, old = ages(text)
    p = {"message_type": t, "flags": flags, "needs": needs, "water_level": None if w == "none" else w,
         "lost_contact": bool(probs["bin"][i, 0] >= th["bin"]["lost_contact"]),
         "insufficient_info": bool(probs["bin"][i, 1] >= th["bin"]["insufficient_info"]),
         "children_ages": kids, "elderly_ages": old}
    p["type_overridden"] = overridden
    p["priority_rules"] = rule_priority(p)
    if t == "help_request":
        pr = probs["prio"][i]
        head = "red" if pr[0] >= th["prio_red"] else PRIO[int(pr[1:].argmax()) + 1]
        p["priority_head"] = head
        p["priority"] = max(p["priority_rules"], head, key=lambda c: RANK[c])
    else:
        p["priority_head"] = p["priority"] = "none"
    # confidence for the Agent's escalate-to-LLM decision: weakest of the decisions that matter most
    margins = [float(probs["type"][i].max())]
    for j, f in enumerate(FLAGS):                             # a red flag near its threshold = uncertain
        if f in RED_FLAGS:
            q = float(probs["flags"][i, j]); thr = th["flags"][f]
            if 0.25 * thr < q < min(0.95, thr + 0.3):
                margins.append(0.5 + abs(q - thr))
    p["confidence"] = round(min(margins), 3)
    p["red_prob"] = round(float(max(probs["prio"][i][0] if t == "help_request" else 0,
                                    max(probs["flags"][i, FLAGS.index(f)] for f in RED_FLAGS))), 3)
    return p


# ---------------- thresholds ----------------
def _prf(y, yhat):
    tp = float((y & yhat).sum()); fp = float((~y & yhat).sum()); fn = float((y & ~yhat).sum())
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)


GRID = [round(x, 2) for x in np.arange(0.05, 0.951, 0.05)]


def tune_thresholds(probs: dict, rows: list[dict], texts: list[str], red_recall: float = 0.98, min_pos: int = 5) -> dict:
    """Pick thresholds on the DEV set. Red flags: highest threshold that still reaches the recall target.
    Others: best F1. Too few positives -> keep 0.5 (a threshold tuned on 2 examples is noise)."""
    th = default_thresholds()
    Y = np.array([encode(r)["flags"] for r in rows]).astype(bool)
    N = np.array([encode(r)["needs"] for r in rows]).astype(bool)
    B = np.array([encode(r)["bin"] for r in rows]).astype(bool)
    for cols, name, Ytrue, P in ((FLAGS, "flags", Y, probs["flags"]), (NEEDS, "needs", N, probs["needs"]), (BIN, "bin", B, probs["bin"])):
        for j, c in enumerate(cols):
            y = Ytrue[:, j]
            if y.sum() < min_pos:
                continue
            if c in RED_FLAGS:
                ok = [t for t in GRID if _prf(y, P[:, j] >= t)[1] >= red_recall]
                th[name][c] = max(ok) if ok else GRID[0]
            else:
                th[name][c] = max(GRID, key=lambda t: (_prf(y, P[:, j] >= t)[2], -abs(t - 0.5)))
    # priority head: highest red threshold for which the FINAL colour reaches the red-recall target
    # + the type-override bar: prefer HIGH values (fewer false alarms) that still reach the target
    gold_red = np.array([r.get("priority") == "red" for r in rows])
    if gold_red.sum() >= min_pos:
        best, best_r = (GRID[0], 0.05), -1.0
        for hm in (0.5, 0.4, 0.3, 0.2, 0.1, 0.05):
            for t in sorted(GRID, reverse=True):
                th2 = {**th, "prio_red": t, "help_min": hm}
                pred_red = np.array([decode(probs, i, th2, texts[i])["priority"] == "red" for i in range(len(rows))])
                r = _prf(gold_red, pred_red)[1]
                if r > best_r:
                    best, best_r = (t, hm), r
                if r >= red_recall:
                    break
            if best_r >= red_recall:
                break
        th["prio_red"], th["help_min"] = best
        th["dev_red_recall"] = round(best_r, 4)
    return th


# ---------------- metrics ----------------
def evaluate(preds: list[dict], rows: list[dict]) -> dict:
    out = {"n": len(rows)}
    out["type_acc"] = round(float(np.mean([p["message_type"] == r["message_type"] for p, r in zip(preds, rows)])), 4)
    out["type"] = {}
    for t in TYPES:
        y = np.array([r["message_type"] == t for r in rows]); yh = np.array([p["message_type"] == t for p in preds])
        P, R, F = _prf(y, yh); out["type"][t] = {"support": int(y.sum()), "precision": round(P, 3), "recall": round(R, 3), "f1": round(F, 3)}
    for key, cols in (("flags", FLAGS), ("needs", NEEDS)):
        out[key] = {}
        for c in cols:
            y = np.array([c in r.get(key, []) for r in rows]); yh = np.array([c in p[key] for p in preds])
            if y.sum() or yh.sum():
                P, R, F = _prf(y, yh); out[key][c] = {"support": int(y.sum()), "precision": round(P, 3), "recall": round(R, 3), "f1": round(F, 3)}
        f1s = [v["f1"] for v in out[key].values() if v["support"] >= 5]
        out[key + "_macro_f1"] = round(float(np.mean(f1s)), 4) if f1s else None
    help_rows = [(p, r) for p, r in zip(preds, rows) if r["message_type"] == "help_request"]
    if help_rows:
        out["water_acc"] = round(float(np.mean([(p["water_level"] or "none") == (r.get("water_level") or "none") for p, r in help_rows])), 4)
        for name in ("priority", "priority_rules", "priority_head"):
            out[name + "_acc"] = round(float(np.mean([p[name] == r.get("priority") for p, r in help_rows])), 4)
        y = np.array([r.get("priority") == "red" for _, r in help_rows]); yh = np.array([p["priority"] == "red" for p, _ in help_rows])
        P, R, F = _prf(y, yh)
        out["red"] = {"support": int(y.sum()), "precision": round(P, 3), "recall": round(R, 3), "f1": round(F, 3)}
        out["priority_confusion"] = {g: {q: sum(1 for p, r in help_rows if r.get("priority") == g and p["priority"] == q) for q in PRIO} for g in PRIO}
        out["under_triage"] = sum(1 for p, r in help_rows if RANK[p["priority"]] < RANK.get(r.get("priority"), -1))
    return out


def score(m: dict) -> float:
    """Model selection on DEV: red recall matters most, then type accuracy, flags F1, colour accuracy."""
    red = (m.get("red") or {}).get("recall", 0)
    return round(0.4 * red + 0.25 * m["type_acc"] + 0.2 * (m.get("flags_macro_f1") or 0) + 0.15 * m.get("priority_acc", 0), 4)


# ---------------- torch part ----------------
try:
    import torch
    import torch.nn as nn
except ImportError:                                     # backend without torch: the numpy helpers above still work
    torch = nn = None

if nn is not None:
    class FloodWCB(nn.Module):
        def __init__(self, backbone, hidden: int, dropout: float = 0.1):
            super().__init__()
            self.backbone = backbone
            self.drop = nn.Dropout(dropout)
            self.heads = nn.ModuleDict({"type": nn.Linear(hidden, len(TYPES)), "flags": nn.Linear(hidden, len(FLAGS)),
                                        "needs": nn.Linear(hidden, len(NEEDS)), "water": nn.Linear(hidden, len(WATER)),
                                        "bin": nn.Linear(hidden, len(BIN)), "prio": nn.Linear(hidden, len(PRIO))})

        def forward(self, input_ids, attention_mask):
            h = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
            m = attention_mask.unsqueeze(-1).to(h.dtype)
            pooled = (h * m).sum(1) / m.sum(1).clamp(min=1.0)          # mean pooling over real tokens
            x = self.drop(pooled)
            return {k: head(x) for k, head in self.heads.items()}

    class Predictor:
        """Load an exported model folder (no internet needed) and predict on raw texts."""
        def __init__(self, path, device: str | None = None, threads: int | None = None):
            from transformers import AutoConfig, AutoModel, AutoTokenizer
            path = Path(path)
            self.cfg = json.loads((path / "floodwcb_config.json").read_text(encoding="utf-8"))
            self.th = self.cfg["thresholds"]
            self.tok = AutoTokenizer.from_pretrained(path)
            backbone = AutoModel.from_config(AutoConfig.from_pretrained(path))
            self.model = FloodWCB(backbone, backbone.config.hidden_size)
            sd = torch.load(path / "model.pt", map_location="cpu")
            self.model.load_state_dict({k: v.float() for k, v in sd.items()})
            self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
            if threads:
                torch.set_num_threads(threads)
            self.model.to(self.device).eval()
            self.max_len = self.cfg.get("max_len", 256)

        @torch.no_grad()
        def logits(self, texts: list[str], batch: int = 16) -> dict:
            out = {k: [] for k in ("type", "flags", "needs", "water", "bin", "prio")}
            for i in range(0, len(texts), batch):
                enc = self.tok([preprocess(t) for t in texts[i:i + batch]], truncation=True, max_length=self.max_len,
                               padding=True, return_tensors="pt").to(self.device)
                o = self.model(enc["input_ids"], enc["attention_mask"])
                for k in out:
                    out[k].append(o[k].float().cpu().numpy())
            return {k: np.concatenate(v) for k, v in out.items()}

        def predict(self, texts: list[str], batch: int = 16) -> list[dict]:
            probs = to_probs(self.logits(texts, batch))
            return [decode(probs, i, self.th, t) for i, t in enumerate(texts)]


In [ ]:
import importlib, floodwcb as W
importlib.reload(W)
print("labels:", len(W.TYPES), "types |", len(W.FLAGS), "flags |", len(W.NEEDS), "needs |", len(W.WATER), "water levels")

## 4. ตรวจข้อมูลก่อนเทรน

In [ ]:
from collections import Counter
load = lambda n: [json.loads(l) for l in open(DATA / n, encoding="utf-8") if l.strip()]
train_rows, dev_rows, test_rows = load("train.jsonl"), load("dev.jsonl"), load("test.jsonl")
print(f"train {len(train_rows)} | dev {len(dev_rows)} | test {len(test_rows)}")
print("ประเภท (train):", dict(Counter(r["message_type"] for r in train_rows)))
print("กลุ่มเปราะบาง (train):", dict(Counter(f for r in train_rows for f in r["flags"]).most_common()))
print("สี (train, help):", dict(Counter(r["priority"] for r in train_rows if r["message_type"] == "help_request")))
reviewed = sum(1 for r in test_rows if r.get("human_reviewed"))
print(f"\nชุดทดสอบที่คนตรวจแล้ว: {reviewed}/{len(test_rows)}")
if reviewed < len(test_rows) * 0.9:
    print("⚠️ ชุดทดสอบยังไม่ได้ให้คนตรวจครบ — ผลที่ได้คือ 'เหมือน Groq แค่ไหน' ไม่ใช่ 'ถูกจริงแค่ไหน' ห้ามใช้ตัวเลขนี้ในรายงาน")
if len(train_rows) < 800:
    print("⚠️ ข้อมูลเทรนน้อย (<800) — เทรนได้ แต่ผลจะยังไม่นิ่ง")
ids = lambda rows: {r["row_id"] for r in rows}
assert not (ids(train_rows) & ids(test_rows)) and not (ids(dev_rows) & ids(test_rows)), "ข้อมูลรั่ว: มีแถวซ้ำระหว่าง train/dev กับ test"

## 5. โหลด WangchanBERTa

In [ ]:
import random, numpy as np, torch, torch.nn as nn
from transformers import AutoTokenizer, AutoModel
SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
try:
    tok = AutoTokenizer.from_pretrained(W.MODEL_NAME)
except Exception as e:
    print("fast tokenizer ใช้ไม่ได้ -> ใช้แบบ slow:", e)
    tok = AutoTokenizer.from_pretrained(W.MODEL_NAME, use_fast=False)
backbone = AutoModel.from_pretrained(W.MODEL_NAME)
model = W.FloodWCB(backbone, backbone.config.hidden_size).to(device)
sample = train_rows[0]["text"][:80]
print("ตัวอย่างการตัดคำ:", tok.tokenize(W.preprocess(sample))[:25])
lens = [len(tok(W.preprocess(r["text"]))["input_ids"]) for r in train_rows[:1000]]
print(f"ความยาว token: median {int(np.median(lens))}, p95 {int(np.percentile(lens, 95))}, ยาวเกิน {MAX_LEN}: {np.mean(np.array(lens) > MAX_LEN):.1%}")

## 6. ชุดข้อมูล + loss (ถ่วงน้ำหนักกลุ่มที่หายาก)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class DS(Dataset):
    def __init__(self, rows):
        self.txt = [W.preprocess(r["text"]) for r in rows]
        self.enc = [W.encode(r) for r in rows]
    def __len__(self): return len(self.txt)
    def __getitem__(self, i): return self.txt[i], self.enc[i]

def collate(batch):
    t = tok([b[0] for b in batch], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
    e = [b[1] for b in batch]
    y = {"type": torch.tensor([x["type"] for x in e]), "water": torch.tensor([x["water"] for x in e]),
         "prio": torch.tensor([x["prio"] for x in e]), "bin_mask": torch.tensor([x["bin_mask"] for x in e], dtype=torch.float32)}
    for k in ("flags", "needs", "bin"):
        y[k] = torch.tensor(np.stack([x[k] for x in e]))
    return {"input_ids": t["input_ids"], "attention_mask": t["attention_mask"]}, y

train_ds, dev_ds, test_ds = DS(train_rows), DS(dev_rows), DS(test_rows)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, collate_fn=collate, num_workers=2)

def pos_weight(key):
    Y = np.stack([e[key] for e in train_ds.enc]); pos = Y.sum(0)
    return torch.tensor(np.clip((len(Y) - pos) / np.maximum(pos, 1), 1, 10), dtype=torch.float32, device=device)
tc = np.bincount([e["type"] for e in train_ds.enc], minlength=len(W.TYPES))
type_w = torch.tensor(np.clip(np.sqrt(tc.sum() / np.maximum(tc, 1) / len(tc)), 0.5, 5), dtype=torch.float32, device=device)
ce_type = nn.CrossEntropyLoss(weight=type_w)
ce = nn.CrossEntropyLoss(ignore_index=W.IGNORE)
bce_flags = nn.BCEWithLogitsLoss(pos_weight=pos_weight("flags"))
bce_needs = nn.BCEWithLogitsLoss(pos_weight=pos_weight("needs"))
bce_bin = nn.BCEWithLogitsLoss(reduction="none")

def loss_fn(o, y):
    L = ce_type(o["type"], y["type"]) + 1.5 * bce_flags(o["flags"], y["flags"]) + bce_needs(o["needs"], y["needs"])
    if (y["water"] != W.IGNORE).any(): L = L + 0.7 * ce(o["water"], y["water"])
    if (y["prio"] != W.IGNORE).any(): L = L + ce(o["prio"], y["prio"])
    m = y["bin_mask"]
    if m.sum() > 0: L = L + 0.5 * (bce_bin(o["bin"], y["bin"]).mean(1) * m).sum() / m.sum()
    return L

@torch.no_grad()
def predict_probs(ds):
    model.eval(); out = {k: [] for k in ("type", "flags", "needs", "water", "bin", "prio")}
    for x, _ in DataLoader(ds, batch_size=64, collate_fn=collate):
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            o = model(x["input_ids"].to(device), x["attention_mask"].to(device))
        for k in out: out[k].append(o[k].float().cpu().numpy())
    return W.to_probs({k: np.concatenate(v) for k, v in out.items()})

def run_eval(ds, rows, th=None):
    probs = predict_probs(ds); texts = [r["text"] for r in rows]
    th = th or W.tune_thresholds(probs, rows, texts, RED_RECALL_TARGET)
    preds = [W.decode(probs, i, th, texts[i]) for i in range(len(rows))]
    return W.evaluate(preds, rows), th, preds, probs
print("type weights:", dict(zip(W.TYPES, type_w.cpu().numpy().round(2))))

## 7. เทรน
แต่ละรอบจะวัดกับชุด dev แล้วเก็บรอบที่ดีที่สุดไว้ (คะแนนเน้น: จับเคสแดงได้ > ประเภทถูก > กลุ่มเปราะบาง > สีถูก)

In [ ]:
import time
from transformers import get_linear_schedule_with_warmup
opt = torch.optim.AdamW([{"params": model.backbone.parameters(), "lr": LR},
                         {"params": model.heads.parameters(), "lr": 1e-3}], weight_decay=0.01)
total = EPOCHS * len(train_dl)
sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
use_amp = device == "cuda"
try:
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

BEST = WORK / "best.pt"; best_score, bad, history = -1, 0, []
for ep in range(1, EPOCHS + 1):
    model.train(); t0, run = time.time(), 0.0
    for step, (x, y) in enumerate(train_dl, 1):
        x = {k: v.to(device) for k, v in x.items()}; y = {k: v.to(device) for k, v in y.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            o = model(x["input_ids"], x["attention_mask"])
        loss = loss_fn({k: v.float() for k, v in o.items()}, y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        run += loss.item()
    m, th, _, _ = run_eval(dev_ds, dev_rows)
    s = W.score(m); history.append({"epoch": ep, "loss": round(run / step, 4), "score": s, "type_acc": m["type_acc"],
                                    "flags_f1": m.get("flags_macro_f1"), "red_recall": (m.get("red") or {}).get("recall"),
                                    "priority_acc": m.get("priority_acc")})
    print(f"epoch {ep}: loss {run / step:.4f} | dev score {s} | type {m['type_acc']} | flags F1 {m.get('flags_macro_f1')} "
          f"| red recall {(m.get('red') or {}).get('recall')} | colour {m.get('priority_acc')} | {time.time() - t0:.0f}s")
    if s > best_score:
        best_score, bad = s, 0
        torch.save(model.state_dict(), BEST)
        if DRIVE_DIR: shutil.copy(BEST, DRIVE_DIR / "best.pt")
        print("   ✓ เก็บเป็นรอบที่ดีที่สุด")
    else:
        bad += 1
        if bad >= PATIENCE:
            print("   หยุด: dev ไม่ดีขึ้น", PATIENCE, "รอบติด"); break

## 8. วัดผลกับชุดทดสอบ (ไม่เคยเห็นตอนเทรน)
เกณฑ์ (thresholds) ถูกตั้งจากชุด dev เท่านั้น แล้วนำมาใช้กับ test ตรง ๆ

In [ ]:
import pandas as pd
model.load_state_dict(torch.load(BEST, map_location=device))
m_dev, TH, _, _ = run_eval(dev_ds, dev_rows)
m_test, _, test_preds, _ = run_eval(test_ds, test_rows, TH)
print("เกณฑ์ที่ได้จาก dev:", json.dumps({"prio_red": TH["prio_red"], "help_min": TH["help_min"],
      "red_flags": {f: TH["flags"][f] for f in W.RED_FLAGS}}, ensure_ascii=False))
summary = {"ประเภทข้อความถูก": m_test["type_acc"], "กลุ่มเปราะบาง (macro F1)": m_test.get("flags_macro_f1"),
           "สิ่งที่ต้องการ (macro F1)": m_test.get("needs_macro_f1"), "ระดับน้ำถูก": m_test.get("water_acc"),
           "สีถูก (รวม 2 ทาง)": m_test.get("priority_acc"), "สีถูก (กฎอย่างเดียว)": m_test.get("priority_rules_acc"),
           "สีถูก (หัวทำนายอย่างเดียว)": m_test.get("priority_head_acc"),
           "จับเคสแดงได้ (recall)": (m_test.get("red") or {}).get("recall"), "เคสแดงที่ทาย ถูกจริง (precision)": (m_test.get("red") or {}).get("precision"),
           "ให้สีต่ำกว่าจริง (เคส)": m_test.get("under_triage")}
display(pd.DataFrame(summary.items(), columns=["ตัววัด (test)", "ค่า"]))
display(pd.DataFrame(m_test["type"]).T.rename_axis("ประเภท"))
display(pd.DataFrame(m_test["flags"]).T.rename_axis("กลุ่มเปราะบาง").sort_values("support", ascending=False))
display(pd.DataFrame(m_test["priority_confusion"]).T.rename_axis("จริง \\ ทาย"))

## 9. ดูข้อผิดพลาด (ใช้ปรับคู่มือติดป้าย / หาข้อมูลเพิ่ม)

In [ ]:
pd.set_option("display.max_colwidth", 160)
err = []
for p, r in zip(test_preds, test_rows):
    if r["message_type"] == "help_request" and W.RANK[p["priority"]] < W.RANK.get(r["priority"], -1):
        err.append({"ชนิด": "ให้สีต่ำกว่าจริง", "จริง": r["priority"], "ทาย": p["priority"], "text": r["text"][:160],
                    "flags จริง": r["flags"], "flags ทาย": p["flags"]})
    elif p["message_type"] != r["message_type"]:
        err.append({"ชนิด": "ประเภทผิด", "จริง": r["message_type"], "ทาย": p["message_type"], "text": r["text"][:160],
                    "flags จริง": r["flags"], "flags ทาย": p["flags"]})
print(f"ผิดทั้งหมด {len(err)} จาก {len(test_rows)}  (ให้สีต่ำกว่าจริง {sum(e['ชนิด'] == 'ให้สีต่ำกว่าจริง' for e in err)})")
display(pd.DataFrame(err).sort_values("ชนิด").head(40))

## 10. ลองพิมพ์ข้อความเอง

In [ ]:
def try_text(*texts):
    ds = DS([{"text": t, "message_type": "irrelevant", "flags": [], "needs": [], "priority": "none"} for t in texts])
    probs = predict_probs(ds)
    for i, t in enumerate(texts):
        p = W.decode(probs, i, TH, t)
        print(f"\n{t}\n -> {p['message_type']} | สี {p['priority']} (กฎ {p['priority_rules']}, หัว {p['priority_head']}) "
              f"| flags {p['flags']} | needs {p['needs']} | น้ำ {p['water_level']} | ติดต่อไม่ได้ {p['lost_contact']} | มั่นใจ {p['confidence']}")

try_text("แม่ต้องไปโรงพยาบาลทุกวันอังคาร ตอนนี้ออกจากบ้านไม่ได้ น้ำถึงชั้น 2 ซ.8 รัตนอุทิศ",
         "ได้รับการช่วยเหลือแล้วค่ะ ขอบคุณทีมกู้ภัยมาก ๆ",
         "สู้ ๆ นะคะพี่กัน",
         "มีเรือว่าง 2 ลำ อยู่หน้า มอ. ใครต้องการติดต่อได้",
         "ลูกสาว 8 เดือน ไข้สูง ติดอยู่ชั้น 2 คลองแห")

## 11. ส่งออกโมเดล
ได้ `floodwcb_model.zip` → แตกไว้ที่ `04_flood_nlp_model/artifacts/floodwcb_model/` ในโปรเจกต์

In [ ]:
import datetime
EXPORT = Path("/content/floodwcb_model"); shutil.rmtree(EXPORT, ignore_errors=True); EXPORT.mkdir()
sd = {k: (v.half() if v.dtype == torch.float32 else v) for k, v in model.state_dict().items()}   # fp16: ไฟล์เล็กลงครึ่งหนึ่ง
torch.save(sd, EXPORT / "model.pt")
tok.save_pretrained(EXPORT); backbone.config.save_pretrained(EXPORT)
shutil.copy("floodwcb.py", EXPORT / "floodwcb.py")
cfg = {"model_name": W.MODEL_NAME, "max_len": MAX_LEN, "thresholds": TH, "version": "wcb-" + datetime.date.today().isoformat(),
       "labels": {"types": W.TYPES, "flags": W.FLAGS, "needs": W.NEEDS, "water": W.WATER, "prio": W.PRIO},
       "data": {"train": len(train_rows), "dev": len(dev_rows), "test": len(test_rows),
                "test_human_reviewed": sum(1 for r in test_rows if r.get("human_reviewed"))},
       "metrics_test": m_test, "metrics_dev": m_dev, "history": history}
(EXPORT / "floodwcb_config.json").write_text(json.dumps(cfg, ensure_ascii=False, indent=2, default=float), encoding="utf-8")

# ตรวจว่าโหลดกลับได้ และทายเหมือนเดิม (แบบเดียวกับที่ backend จะโหลด)
P = W.Predictor(EXPORT, device="cpu")
check = [r["text"] for r in dev_rows[:20]]
same = sum(a["message_type"] == b["message_type"] and a["priority"] == b["priority"]
           for a, b in zip(P.predict(check), [W.decode(predict_probs(DS(dev_rows[:20])), i, TH, t) for i, t in enumerate(check)]))
print(f"โหลดกลับแล้วทายตรงกัน {same}/20 (ต่างกันเล็กน้อยได้จาก fp16)")

shutil.make_archive("/content/floodwcb_model", "zip", EXPORT)
size = Path("/content/floodwcb_model.zip").stat().st_size / 1e6
print(f"floodwcb_model.zip {size:.0f} MB")
if DRIVE_DIR:
    shutil.copy("/content/floodwcb_model.zip", DRIVE_DIR / "floodwcb_model.zip"); print("เซฟลง Drive แล้ว:", DRIVE_DIR)
try:
    from google.colab import files; files.download("/content/floodwcb_model.zip")
except Exception as e:
    print("ดาวน์โหลดเองจากแถบไฟล์ด้านซ้าย:", e)